# Building a CNN in PyTorch

_PyTorch_

**Wire conv, pool, and a linear head into a small image classifier, then train it.**

A CNN (Convolutional Neural Network) is a stack of layers that turn an image into a class.

       The first layers are convolutional blocks: Conv2d finds local patterns, BatchNorm2d steadies the signal, ReLU adds a nonlinearity, and MaxPool2d shrinks the map. Each block makes the picture smaller in space but richer in channels.

---

This notebook is a practice scaffold for the **Building a CNN in PyTorch** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Build the data pipeline

We normalize each MNIST image so its pixels have zero mean and unit variance, which keeps the optimizer well behaved. A `DataLoader` then wraps the dataset and hands the model shuffled mini-batches shaped `(N, C, H, W)` — the 4-D layout PyTorch convolutions expect.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Normalize, then a DataLoader feeds batches of (N, C, H, W).
tfm = transforms.Compose([
    transforms.ToTensor(),                       # uint8 0..255 -> float 0..1, shape (1, 28, 28)
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean/std -> zero mean, unit variance
])

train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=256, shuffle=False)

### Step 2 — Define the CNN

The model is two convolutional blocks followed by a linear head. Each block does `Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d`: the conv finds local patterns, BatchNorm steadies the signal, ReLU adds nonlinearity, and the pool halves the spatial size (28 -> 14 -> 7). After the blocks we flatten the `32 x 7 x 7` map into a `1568`-vector and the `Linear` head emits raw class logits — no softmax, because the loss applies it internally.

In [ ]:
# Model: 2 conv blocks + a Linear head (LeNet-style).
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),  # (N,1,28,28) -> (N,16,28,28)
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),                                       # -> (N,16,14,14)
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), # -> (N,32,14,14)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                                       # -> (N,32,7,7)
        )
        # flattened size = channels * H * W = 32 * 7 * 7 = 1568
        self.head = nn.Linear(32 * 7 * 7, num_classes)            # outputs raw logits (no softmax!)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)   # keep batch dim, flatten the rest -> (N, 1568)
        return self.head(x)

model = SmallCNN().to(device)
loss_fn = nn.CrossEntropyLoss()                 # expects logits + integer class labels
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### Step 3 — Train the model

Each step follows the standard PyTorch loop: zero the gradients, run the forward pass to get logits, compute the loss, backprop to fill `.grad`, then let the optimizer update the weights. `model.train()` puts BatchNorm into batch-statistics mode, and `loss.item()` detaches the scalar so no graph is held across steps.

In [ ]:
# Training loop.
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()                               # BatchNorm uses batch stats now
    running = 0.0
    for images, labels in train_dl:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()                   # clear grads (they accumulate otherwise)
        logits = model(images)                  # forward
        loss = loss_fn(logits, labels)
        loss.backward()                         # backprop fills .grad
        optimizer.step()                        # update weights
        running += loss.item()                  # .item() detaches -> no graph kept across steps
    print(f"epoch {epoch+1}/{EPOCHS}  avg train loss {running/len(train_dl):.4f}")

### Step 4 — Evaluate on the test set

For evaluation we switch to `model.eval()` so BatchNorm uses its stored running statistics instead of the batch's, and wrap the loop in `torch.no_grad()` to skip graph building. We take the `argmax` over the logits as the predicted class and tally how many match the true labels.

In [ ]:
# Evaluation.
model.eval()                                    # BatchNorm switches to running stats
correct = total = 0
with torch.no_grad():                           # no graph at inference -> less memory
    for images, labels in test_dl:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"test accuracy: {correct/total:.4f}  on {total} images")

## Visualize the data & results

_On the same image task, does a CNN really beat a plain MLP on flattened pixels?_

### Step 1 — Load the digits and shape them for convs

We use scikit-learn's 1797 real 8x8 digit images, scale the pixels to `0..1`, and split into train/test. PyTorch convolutions need a channel axis, so `unsqueeze(1)` turns each `(N, 8, 8)` batch into `(N, 1, 8, 8)`. A `DataLoader` over a `TensorDataset` then feeds shuffled mini-batches.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

d = load_digits()                                  # 1797 real 8x8 digits
X = (d.images / 16.0).astype("float32")            # scale pixels to 0..1, shape (N, 8, 8)
y = d.target.astype("int64")

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# Add channel axis -> (N, 1, 8, 8); PyTorch convs want (N, C, H, W).
Xtr_t = torch.tensor(Xtr).unsqueeze(1)
Xte_t = torch.tensor(Xte).unsqueeze(1)
ytr_t = torch.tensor(ytr)
yte_t = torch.tensor(yte)

dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=64, shuffle=True)

### Step 2 — Define a CNN and an MLP to compare

To test whether convolutions actually help, we build two models. The `CNN` keeps the 2-D structure: two conv blocks (`Conv -> BatchNorm -> ReLU -> MaxPool`) shrink `8x8 -> 4x4 -> 2x2` while growing channels, then a `Linear` head reads the flattened `16*2*2 = 64` features. The `MLP` throws the spatial layout away — it flattens the 64 raw pixels straight into dense layers.

In [ ]:
class CNN(nn.Module):                              # two conv blocks + Linear head
    def __init__(s):
        super().__init__()
        s.c1 = nn.Conv2d(1, 8, 3, padding=1)
        s.b1 = nn.BatchNorm2d(8)
        s.c2 = nn.Conv2d(8, 16, 3, padding=1)
        s.b2 = nn.BatchNorm2d(16)
        s.pool = nn.MaxPool2d(2)
        s.relu = nn.ReLU()
        s.fc = nn.Linear(16 * 2 * 2, 10)
    def forward(s, x):
        x = s.pool(s.relu(s.b1(s.c1(x))))          # 8x8 -> 4x4
        x = s.pool(s.relu(s.b2(s.c2(x))))          # 4x4 -> 2x2
        return s.fc(x.flatten(1))                   # 16*2*2 = 64 -> 10 logits

class MLP(nn.Module):                              # flat pixels -> dense layers
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(s, x):
        return s.net(x.flatten(1))

### Step 3 — Train both and compare accuracy

We train each model with the same recipe (Adam, cross-entropy, 12 epochs) from the same seed so the comparison is fair, then measure test accuracy. The CNN, which exploits the spatial structure of the digits, should come out ahead of the MLP on flattened pixels.

In [ ]:
def train(model, epochs=12):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            loss = lf(model(xb), yb)
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return (model(Xte_t).argmax(1) == yte_t).float().mean().item()

torch.manual_seed(0)
cnn_acc = train(CNN())
torch.manual_seed(0)
mlp_acc = train(MLP())

print(round(cnn_acc, 4), round(mlp_acc, 4))        # -> 0.963 0.9315

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. The channel-axis pitfall. Make a grayscale batch
            arr = torch.randn(64, 28, 28). Try nn.Conv2d(1, 8, 3)(arr) and observe the shape
            error, then add the channel axis with arr.unsqueeze(1) and run it. Predict the output shape
            before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Insert C=1 with arr.unsqueeze(1) to get (64, 1, 28, 28). — _nn.Conv2d wants (N, C, H, W) — an explicit channel axis even for grayscale._
- Run the conv and print out.shape. — _A 3&times;3 conv with no padding trims the border, so 28&rarr;26._

**Answer:** import torch, torch.nn as nn
arr = torch.randn(64, 28, 28)
# nn.Conv2d(1, 8, 3)(arr)  -> RuntimeError: expected 4D input (N,C,H,W)
x = arr.unsqueeze(1)              # (64, 1, 28, 28)
out = nn.Conv2d(1, 8, 3)(x)
print(out.shape)                  # torch.Size([64, 8, 26, 26])

</details>

**Problem 2.** Type this in Colab. Verify the conv output-shape formula. Feed x = torch.randn(2, 3, 32, 32)
            through (a) nn.Conv2d(3, 16, kernel_size=3, padding=1) and (b)
            nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=0). Predict each output's H and W from
            $H_{out}=\lfloor (H+2p-k)/s \rfloor + 1$, then verify.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply both convs and print .shape. — _Padded 3&times;3 (p=1,s=1) keeps 32; the 5&times;5 stride-2 gives (32-5)/2+1 = 14._
- Check the channel dim is the conv's out_channels (16). — _out_channels is how many filters/feature maps the layer learns._

**Answer:** x = torch.randn(2, 3, 32, 32)
a = nn.Conv2d(3, 16, kernel_size=3, padding=1)(x)
b = nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=0)(x)
print(a.shape)   # torch.Size([2, 16, 32, 32])  (p=1,s=1 keeps size)
print(b.shape)   # torch.Size([2, 16, 14, 14])  ((32-5)/2 + 1 = 14)

</details>

**Problem 3.** Type this in Colab. Trace a two-block feature extractor. Build
            nn.Sequential(Conv2d(1,16,3,padding=1), ReLU(), MaxPool2d(2), Conv2d(16,32,3,padding=1), ReLU(), MaxPool2d(2))
            and run an MNIST-sized input torch.randn(4, 1, 28, 28) through it. Predict the final shape,
            then compute the flattened feature count.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Print the output shape after the two conv+pool blocks. — _Each MaxPool2d(2) halves H and W: 28&rarr;14&rarr;7, with 32 channels._
- Compute 32 * 7 * 7 for the flattened size. — _That is the input width the first Linear head needs._

**Answer:** feat = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)
out = feat(torch.randn(4, 1, 28, 28))
print(out.shape)            # torch.Size([4, 32, 7, 7])
print(32 * 7 * 7)           # 1568  -> Linear(1568, num_classes)

</details>

**Problem 4.** Type this in Colab. The flatten-size pitfall. Build a tiny CNN where the head is wrongly
            nn.Linear(100, 10) after the two-block extractor above. Feed a (4, 1, 28, 28) batch
            and read the mat1 and mat2 shapes cannot be multiplied error. Fix the head to the correct size.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Flatten with torch.flatten(x, 1) (keep the batch dim) before the Linear. — _The head is a matrix multiply; the in-features must equal channels&times;H&times;W = 1568._
- Replace Linear(100, 10) with Linear(1568, 10). — _Matching the flattened size removes the shape-mismatch error._

**Answer:** class Net(nn.Module):
    def __init__(s, in_feats):
        super().__init__()
        s.feat = feat                       # the 2-block extractor above
        s.head = nn.Linear(in_feats, 10)
    def forward(s, x):
        return s.head(torch.flatten(s.feat(x), 1))

x = torch.randn(4, 1, 28, 28)
# Net(100)(x)   -> RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x1568 and 100x10)
print(Net(1568)(x).shape)    # torch.Size([4, 10])  -- correct flattened size

</details>

**Problem 5.** Type this in Colab. Count a Conv2d's parameters. Build
            conv = nn.Conv2d(3, 16, kernel_size=3) and print
            sum(p.numel() for p in conv.parameters()). Predict it first:
            weights = out&times;in&times;k&times;k, plus one bias per output channel.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute weights 16 * 3 * 3 * 3 = 432 plus 16 biases. — _A Conv2d learns one k&times;k filter per (in, out) channel pair, plus a per-output bias._
- Verify with sum(p.numel() for p in conv.parameters()). — _.numel() totals every weight and bias the layer holds._

**Answer:** conv = nn.Conv2d(3, 16, kernel_size=3)
print(sum(p.numel() for p in conv.parameters()))   # 448
# weights 16*3*3*3 = 432, biases 16  ->  432 + 16 = 448

</details>

**Problem 6.** Type this in Colab. The eval-mode pitfall with BatchNorm. Build
            bn = nn.BatchNorm2d(4) and a fixed input x = torch.randn(8, 4, 6, 6). Print
            bn(x).mean().item() in bn.train() mode, then in bn.eval() mode, and
            note they differ.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run the same input through bn.train() then bn.eval(). — _Train mode normalizes with the batch's own stats; eval mode uses stored running stats._
- Observe the two outputs differ. — _That difference is exactly why forgetting model.eval() at test time corrupts the numbers._

**Answer:** torch.manual_seed(0)
bn = nn.BatchNorm2d(4)
x = torch.randn(8, 4, 6, 6)
bn.train(); print(round(bn(x).mean().item(), 4))   # ~ 0.0    (batch stats -> centered)
bn.eval();  print(round(bn(x).mean().item(), 4))   # ~ 0.0276 (running stats -> not centered)

</details>

**Problem 7.** Type this in Colab. Run one full training step of a small CNN. With torch.manual_seed(0),
            build the Net(1568) from before, CrossEntropyLoss (raw logits, no softmax),
            Adam(lr=1e-3), a batch x = torch.randn(8, 1, 28, 28), and labels
            y = torch.randint(0, 10, (8,)). Do zero_grad &rarr; forward &rarr; loss &rarr; backward &rarr;
            step and print the loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pass raw logits to CrossEntropyLoss with integer labels. — _The loss applies log-softmax internally — no softmax layer on the head._
- Run opt.zero_grad() before loss.backward(). — _Gradients accumulate across batches; zeroing keeps each step's update correct._

**Answer:** torch.manual_seed(0)
model = Net(1568)
crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
x = torch.randn(8, 1, 28, 28)
y = torch.randint(0, 10, (8,))
opt.zero_grad()
loss = crit(model(x), y)     # logits + integer labels
loss.backward()
opt.step()
print(round(loss.item(), 4))   # ~ 2.45  (near ln(10)=2.30 at init)

</details>

**Problem 8.** Type this in Colab. Assemble a clean LeNet-style model class end-to-end and confirm the output shape.
            Subclass nn.Module with two Conv2d&rarr;BatchNorm2d&rarr;ReLU&rarr;MaxPool2d blocks
            (1&rarr;16&rarr;32 channels, padding=1) and a Linear(1568, 10) head. Feed
            torch.randn(5, 1, 28, 28) and print the logits shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Put the conv blocks in self.features and the head in self.head. — _Separating the feature extractor from the classifier head is the standard CNN layout._
- In forward, torch.flatten(x, 1) before the head. — _Flattening keeps the batch dim and turns the 32&times;7&times;7 map into the 1568-vector the head expects._

**Answer:** class SmallCNN(nn.Module):
    def __init__(s, num_classes=10):
        super().__init__()
        s.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        )
        s.head = nn.Linear(32 * 7 * 7, num_classes)   # raw logits (no softmax)
    def forward(s, x):
        return s.head(torch.flatten(s.features(x), 1))

print(SmallCNN()(torch.randn(5, 1, 28, 28)).shape)   # torch.Size([5, 10])

</details>